In [0]:


spark.conf.set("fs.azure.account.auth.type.nyctaxisid.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.nyctaxisid.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.nyctaxisid.dfs.core.windows.net", "7c4b429b-ed6b-43f3-a***********8*****929d")
spark.conf.set("fs.azure.account.oauth2.client.secret.nyctaxisid.dfs.core.windows.net", "vwA8Q~DhVr********************p")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.nyctaxisid.dfs.core.windows.net", "https://login.microsoftonline.com/36c5071****************1dbf1/oauth2/token")

CREATE DATABASE

In [0]:
%sql
create database GOLD

DATA READING AND WRITING (DELTA)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


DATA ZONE

In [0]:
df_zone = spark.read.format("parquet")\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load("abfss://silver@nyctaxisid.dfs.core.windows.net/trip_zone")
df_zone.display()

LocationID,Borough,Zone,service_zone,ZONE 1,ZONE 2,Zone 3
1,EWR,Newark Airport,EWR,Newark Airport,null,null
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens,null
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth,null
7,Queens,Astoria,Boro Zone,Astoria,null,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null,null
9,Queens,Auburndale,Boro Zone,Auburndale,null,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null,null


In [0]:
df_zone.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_work.gold.trip_zone")

TRIP DATA

In [0]:
df_trip = spark.read.format("parquet")\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load("abfss://silver@nyctaxisid.dfs.core.windows.net/trips2023")
df_trip.display()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,Trip_Date,Trip_Year,Trip_Month,Hours
2,2023-03-01T00:25:10Z,2023-03-01T00:35:47Z,N,1,82,196,1,2.36,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,-6.5,-1.0,-0.5,0.0,0.0,null,-1.0,-9.0,3,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,6.5,1.0,0.5,0.0,0.0,null,1.0,9.0,3,1,0.0,2023-03-01,2023,3,0
2,2023-02-28T22:59:46Z,2023-02-28T23:08:38Z,N,1,166,74,1,1.66,11.4,1.0,0.5,2.78,0.0,null,1.0,16.68,1,1,0.0,2023-02-28,2023,2,22
2,2023-03-01T00:54:03Z,2023-03-01T01:03:14Z,N,1,236,229,1,3.14,15.6,1.0,0.5,4.17,0.0,null,1.0,25.02,1,1,2.75,2023-03-01,2023,3,0
2,2023-03-01T01:00:09Z,2023-03-01T01:14:37Z,N,1,75,235,1,5.69,23.3,1.0,0.5,4.0,0.0,null,1.0,29.8,1,1,0.0,2023-03-01,2023,3,1
2,2023-03-01T00:09:45Z,2023-03-01T00:26:06Z,N,1,260,160,1,2.92,17.7,1.0,0.5,4.04,0.0,null,1.0,24.24,1,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:39:30Z,2023-03-01T00:39:33Z,N,5,95,264,5,0.0,35.0,0.0,0.0,1.0,0.0,null,1.0,37.0,1,2,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:03:07Z,2023-03-01T00:14:44Z,N,1,244,41,1,3.34,16.3,1.0,0.5,5.64,0.0,null,1.0,24.44,1,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:42:56Z,2023-03-01T00:49:57Z,N,1,83,7,5,1.75,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0,2023-03-01,2023,3,0


In [0]:
df_trip.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@nyctaxisid.dfs.core.windows.net/trips2023")

In [0]:
%sql
select * from nyc_taxi_work.gold.trip_zone

LocationID,Borough,Zone,service_zone,ZONE_1,ZONE_2,Zone_3
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens,null
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth,null
7,Queens,Astoria,Boro Zone,Astoria,null,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null,null
9,Queens,Auburndale,Boro Zone,Auburndale,null,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null,null
11,Brooklyn,Bath Beach,Boro Zone,Bath Beach,null,null


In [0]:
%sql
update nyc_taxi_work.gold.trip_zone
set Borough = 'EMR' where LocationID = '1'

num_affected_rows
11


In [0]:
%sql
delete from nyc_taxi_work.gold.trip_zone
where LocationID = '1'
    


num_affected_rows
11


versioning

In [0]:
%sql
DESCRIBE HISTORY nyc_taxi_work.gold.trip_zone

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-14T22:24:22Z,149492290042880,siddharth.k5890@gmail.com,DELETE,"Map(predicate -> [""(LocationID#22578 = 1)""])",null,List(4393648911353605),1114-181656-2itr090i,5,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2862, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 467, numDeletionVectorsUpdated -> 0, numDeletedRows -> 11, scanTimeMs -> 374, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 92)",null,Databricks-Runtime/14.3.x-photon-scala2.12
5,2026-11-14T22:22:44Z,149492290042880,siddharth.k5890@gmail.com,UPDATE,"Map(predicate -> [""(LocationID#21581 = 1)""])",null,List(4393648911353605),1114-181656-2itr090i,4,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2862, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1494, numDeletionVectorsUpdated -> 0, scanTimeMs -> 607, numAddedFiles -> 1, numUpdatedRows -> 11, numAddedBytes -> 2862, rewriteTimeMs -> 885)",null,Databricks-Runtime/14.3.x-photon-scala2.12
4,2026-11-14T22:21:55Z,149492290042880,siddharth.k5890@gmail.com,UPDATE,"Map(predicate -> [""(LocationID#20373 = 1)""])",null,List(4393648911353605),1114-181656-2itr090i,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3391, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1435, numAddedFiles -> 1, numUpdatedRows -> 11, numAddedBytes -> 2862, rewriteTimeMs -> 1903)",null,Databricks-Runtime/14.3.x-photon-scala2.12
3,2026-11-14T22:17:12Z,149492290042880,siddharth.k5890@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], description -> null, isManaged -> true, properties -> {""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-7e044863-284b-479c-beea-fe0ee0fd4ecd"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-82091b2b-1288-42da-abc5-79d39144fd33""}, statsOnLoad -> true)",null,List(4393648911353605),1114-181656-2itr090i,2,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 2915, numOutputBytes -> 17077)",null,Databricks-Runtime/14.3.x-photon-scala2.12
2,2026-11-14T22:07:31Z,149492290042880,siddharth.k5890@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], description -> null, isManaged -> true, properties -> {""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-7e044863-284b-479c-beea-fe0ee0fd4ecd"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-82091b2b-1288-42da-abc5-79d39144fd33""}, statsOnLoad -> true)",null,List(4393648911353605),1114-181656-2itr090i,1,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 2915, numOutputBytes -> 17077)",null,Databricks-Runtime/14.3.x-photon-scala2.12
1,2026-11-14T22:05:15Z,149492290042880,siddharth.k5890@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], description -> null, isManaged -> true, properties -> {""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.enableRowTracking"":""tru

TIME TRAVEL

In [0]:
%sql
RESTORE nyc_taxi_work.gold.trip_zone TO VERSION AS OF 0

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
50761,4,1,4,17077,50761


In [0]:
%sql
SELECT * FROM nyc_taxi_work.gold.trip_zone

LocationID,Borough,Zone,service_zone,ZONE_1,ZONE_2,Zone_3
1,EWR,Newark Airport,EWR,Newark Airport,null,null
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens,null
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth,null
7,Queens,Astoria,Boro Zone,Astoria,null,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null,null
9,Queens,Auburndale,Boro Zone,Auburndale,null,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null,null


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

silver_path = "abfss://silver@nyctaxisid.dfs.core.windows.net"

df_trip = spark.read.format("parquet") \
    .load(f"{silver_path}/trips2023")

df_zone = spark.read.format("parquet") \
    .load(f"{silver_path}/trip_zone")

display(df_trip)
display(df_zone)

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,Trip_Date,Trip_Year,Trip_Month,Hours
2,2023-03-01T00:25:10Z,2023-03-01T00:35:47Z,N,1,82,196,1,2.36,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,-6.5,-1.0,-0.5,0.0,0.0,null,-1.0,-9.0,3,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,6.5,1.0,0.5,0.0,0.0,null,1.0,9.0,3,1,0.0,2023-03-01,2023,3,0
2,2023-02-28T22:59:46Z,2023-02-28T23:08:38Z,N,1,166,74,1,1.66,11.4,1.0,0.5,2.78,0.0,null,1.0,16.68,1,1,0.0,2023-02-28,2023,2,22
2,2023-03-01T00:54:03Z,2023-03-01T01:03:14Z,N,1,236,229,1,3.14,15.6,1.0,0.5,4.17,0.0,null,1.0,25.02,1,1,2.75,2023-03-01,2023,3,0
2,2023-03-01T01:00:09Z,2023-03-01T01:14:37Z,N,1,75,235,1,5.69,23.3,1.0,0.5,4.0,0.0,null,1.0,29.8,1,1,0.0,2023-03-01,2023,3,1
2,2023-03-01T00:09:45Z,2023-03-01T00:26:06Z,N,1,260,160,1,2.92,17.7,1.0,0.5,4.04,0.0,null,1.0,24.24,1,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:39:30Z,2023-03-01T00:39:33Z,N,5,95,264,5,0.0,35.0,0.0,0.0,1.0,0.0,null,1.0,37.0,1,2,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:03:07Z,2023-03-01T00:14:44Z,N,1,244,41,1,3.34,16.3,1.0,0.5,5.64,0.0,null,1.0,24.44,1,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:42:56Z,2023-03-01T00:49:57Z,N,1,83,7,5,1.75,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0,2023-03-01,2023,3,0


LocationID,Borough,Zone,service_zone,ZONE 1,ZONE 2,Zone 3
1,EWR,Newark Airport,EWR,Newark Airport,null,null
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens,null
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth,null
7,Queens,Astoria,Boro Zone,Astoria,null,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null,null
9,Queens,Auburndale,Boro Zone,Auburndale,null,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null,null


In [0]:
print("Trip columns:", df_trip.columns)
print("Zone columns:", df_zone.columns)

Trip columns: ['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge', 'Trip_Date', 'Trip_Year', 'Trip_Month', 'Hours']
Zone columns: ['LocationID', 'Borough', 'Zone', 'service_zone', 'ZONE 1', 'ZONE 2', 'Zone 3']


Prepare the Gold trips DataFrame

In [0]:
df_gold_trips = df_trip \
    .withColumn(
        "Trip_Date",
        to_date(col("lpep_pickup_datetime"))
    ) \
    .withColumn(
        "Trip_Year",
        year(col("lpep_pickup_datetime"))
    ) \
    .withColumn(
        "Trip_Month",
        month(col("lpep_pickup_datetime"))
    ) \
    .withColumn(
        "Pickup_Hour",
        hour(col("lpep_pickup_datetime"))
    ) \
    .withColumn(
        "Trip_Duration_Minutes",
        round(
            (unix_timestamp(col("lpep_dropoff_datetime")) -
             unix_timestamp(col("lpep_pickup_datetime"))) / 60,
            2
        )
    )

In [0]:
df_gold_trips = df_gold_trips.filter(
    col("lpep_pickup_datetime").isNotNull() &
    col("lpep_dropoff_datetime").isNotNull() &
    col("PULocationID").isNotNull() &
    col("DOLocationID").isNotNull() &
    (col("trip_distance") > 0) &
    (col("fare_amount") >= 0) &
    (col("total_amount") >= 0) &
    (col("lpep_dropoff_datetime") > col("lpep_pickup_datetime"))
).dropDuplicates()

In [0]:
df_gold_trips.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_work.gold.trips2023")

Vendor performance

In [0]:
gold_vendor = df_gold_trips.groupBy("VendorID").agg(
    count("*").alias("Total_Trips"),
    round(sum("total_amount"), 2).alias("Total_Revenue"),
    round(avg("fare_amount"), 2).alias("Average_Fare"),
    round(avg("trip_distance"), 2).alias("Average_Distance"),
    round(avg("Trip_Duration_Minutes"), 2).alias("Average_Duration")
)

In [0]:
display(gold_vendor)

VendorID,Total_Trips,Total_Revenue,Average_Fare,Average_Distance,Average_Duration
1,85157,1861753.74,16.42,2.83,14.42
2,660893,1.58756203E7,18.34,22.03,20.38


In [0]:
gold_vendor.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_work.gold.trips_by_vendor")

Daily revenue metrics

In [0]:
gold_daily = df_gold_trips.groupBy("Trip_Date").agg(
    count("*").alias("Total_Trips"),
    round(sum("total_amount"), 2).alias("Daily_Revenue"),
    round(avg("fare_amount"), 2).alias("Average_Fare"),
    round(avg("trip_distance"), 2).alias("Average_Distance"),
    round(avg("Trip_Duration_Minutes"), 2).alias("Average_Duration")
).orderBy("Trip_Date")

In [0]:
display(gold_daily)

Trip_Date,Total_Trips,Daily_Revenue,Average_Fare,Average_Distance,Average_Duration
2008-12-31,2,51.0,18.4,2.64,539.27
2009-01-01,2,65.75,25.4,4.96,712.05
2022-12-09,2,52.4,20.75,5.43,24.51
2023-01-01,1390,32911.82,18.56,3.73,21.08
2023-01-02,1489,32055.06,16.8,3.04,17.85
2023-01-03,2003,42722.15,16.03,2.72,17.97
2023-01-04,2248,46617.89,15.61,2.61,15.66
2023-01-05,2294,51371.6,17.05,6.53,18.99
2023-01-06,2420,52115.22,16.3,20.75,16.74
2023-01-07,2042,42882.68,16.41,2.91,17.17


In [0]:
gold_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_work.gold.daily_metrics")

Hourly demand analysis

In [0]:
gold_hourly = df_gold_trips.groupBy("Pickup_Hour").agg(
    count("*").alias("Total_Trips"),
    round(sum("total_amount"), 2).alias("Total_Revenue"),
    round(avg("fare_amount"), 2).alias("Average_Fare")
).orderBy("Pickup_Hour")

In [0]:
display(gold_hourly)

Pickup_Hour,Total_Trips,Total_Revenue,Average_Fare
0,13883,335114.61,18.87
1,9607,232226.85,18.98
2,6828,176694.52,20.64
3,5441,153372.69,22.8
4,4277,129849.56,24.64
5,4185,128570.62,24.38
6,10964,259862.77,18.64
7,27619,625790.08,17.79
8,34778,809034.16,18.24
9,38052,893187.16,18.35


In [0]:
gold_hourly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_work.gold.hourly_metrics")

visual


In [0]:
%sql
SELECT
    CAST(VendorID AS STRING) AS Vendor,
    Total_Trips,
    Total_Revenue,
    Average_Fare
FROM nyc_taxi_work.gold.trips_by_vendor
ORDER BY Total_Trips DESC;

Vendor,Total_Trips,Total_Revenue,Average_Fare
2,660893,1.58756203E7,18.34
1,85157,1861753.74,16.42


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    Trip_Date,
    Daily_Revenue
FROM nyc_taxi_work.gold.daily_metrics
ORDER BY Trip_Date;

Trip_Date,Daily_Revenue
2008-12-31,51.0
2009-01-01,65.75
2022-12-09,52.4
2023-01-01,32911.82
2023-01-02,32055.06
2023-01-03,42722.15
2023-01-04,46617.89
2023-01-05,51371.6
2023-01-06,52115.22
2023-01-07,42882.68


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    Trip_Date,
    Daily_Revenue
FROM nyc_taxi_work.gold.daily_metrics
WHERE Trip_Date >= DATE '2023-01-01'
  AND Trip_Date < DATE '2024-01-01'
ORDER BY Trip_Date;

Trip_Date,Daily_Revenue
2023-01-01,32911.82
2023-01-02,32055.06
2023-01-03,42722.15
2023-01-04,46617.89
2023-01-05,51371.6
2023-01-06,52115.22
2023-01-07,42882.68
2023-01-08,31951.54
2023-01-09,45076.87
2023-01-10,44670.98


Databricks visualization. Run in Databricks to view.